# Demo 2 - Emit token metrics (Azure API Management)

## Scenario & talk track

**Metering: design dimensions before dashboards.** In Demo 1 we governed token consumption. In Demo 2 we meter it so the same APIM instance can support showback, chargeback, and operational dashboards.

The four grains to teach from Module 8 are:

1. **Identity grain** -- API ID + Subscription ID (the defaults).
2. **Business grain** -- ClientApp or CostCenter (**bounded values only**).
3. **Metric grain** -- prompt + completion + total (provider dependent).
4. **Cardinality budget** -- ≤5 custom dimensions, ~100 values each in APIM.

`Metric series = values(dim₁) × values(dim₂) × …`

Design dimensions *before* the first dashboard, not after cardinality explodes.

The policy we will apply publishes prompt, completion, and total token metrics into the `module8` namespace and splits them by `API ID`, `Subscription ID`, and a bounded `ClientApp` header:

```xml
<llm-emit-token-metric namespace="module8">
  <dimension name="API ID" />
  <dimension name="Subscription ID" />
  <dimension name="ClientApp" value="@(context.Request.Headers.GetValueOrDefault(&quot;x-client-app&quot;,&quot;unknown&quot;))" />
</llm-emit-token-metric>
```


## Demo isolation

Demo 2 reuses the **same Azure API Management instance** and the same Azure OpenAI / Microsoft Foundry backend values from Demo 1. It does **not** create a new APIM instance.

Isolation comes from:

- Dedicated product: `demo2-metering`
- Dedicated subscription: `demo2-metering-sub`
- Dedicated API ID: `demo2-metering-api`
- Dedicated backend: `demo2-openai-backend`
- The current run time window (captured below) and the persisted `DEMO_RUN` suffix sent as `x-demo-run` for request traceability.

We keep the deck's metric design accurate by **not** putting the run suffix in the `ClientApp` metric dimension. `ClientApp` remains a bounded business grain with exactly the values we allow in the client code.


In [ ]:
import sys
sys.path.append("..")

from datetime import datetime, timedelta, timezone
import re
import time
import uuid

import pandas as pd
import requests

from shared import auth, config, apim, display

cfg = config.load_config(interactive=True)
config.validate_config(cfg)

DEMO_API_ID = "demo2-metering-api"
DEMO_BACKEND_ID = "demo2-openai-backend"
DEMO_PRODUCT_ID = "demo2-metering"
DEMO_SUBSCRIPTION_ID = "demo2-metering-sub"
DEMO_NAMED_VALUE_KEY = "demo2-aoai-key"
DEMO_PATH = "demo2-metering"
DEMO_LOGGER_ID = "demo2-application-insights"
METRIC_NAMESPACE = "module8"
CLIENT_APP_ALLOW_LIST = {"claims-portal", "analyst-copilot"}
TOKEN_METRIC_CANDIDATES = {
    "prompt_tokens": ["prompt_tokens", "Prompt Tokens", "PromptTokens"],
    "completion_tokens": ["completion_tokens", "Completion Tokens", "CompletionTokens"],
    "total_tokens": ["total_tokens", "Total Tokens", "TotalTokens"],
}

DEMO_RUN = cfg.demo_run
RUN_STARTED_AT = datetime.now(timezone.utc)
API_STYLE = cfg.aoai_api_style

print(f"Demo 2 APIM product/subscription: {DEMO_PRODUCT_ID} / {DEMO_SUBSCRIPTION_ID}")
print(f"Azure OpenAI endpoint: {cfg.aoai_endpoint} (api style: {API_STYLE})")
print(f"Current DEMO_RUN suffix sent as x-demo-run: {DEMO_RUN}")
print(f"Metric namespace: {METRIC_NAMESPACE}")
print(f"Bounded ClientApp values: {sorted(CLIENT_APP_ALLOW_LIST)}")


## Preflight checks (run these out loud)

Before configuring or sending traffic, call out each prerequisite:

1. **Application Insights connected** -- APIM has an Application Insights logger, or this notebook has enough `.env` values to create one.
2. **LLM API logging enabled** -- the API diagnostic has LLM / large-language-model logging settings.
3. **Custom metrics with dimensions enabled** -- App Insights **Enable alerting on custom metric dimensions** / usage-and-estimated-costs setting. If ARM cannot detect this reliably, follow the manual portal instruction shown below.
4. **Client sends a bounded `x-client-app` value** -- the notebook enforces an allow-list before sending traffic.


In [ ]:
display.header("Preflight checks")

app_insights_resource_id = cfg.app_insights_resource_id
if not app_insights_resource_id and cfg.app_insights_name:
    app_insights_resource_id = apim.app_insights_resource_id(
        cfg.subscription_id, cfg.resource_group, cfg.app_insights_name
    )

existing_ai_logger = apim.get_app_insights_for_apim(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name
)
if not app_insights_resource_id and existing_ai_logger:
    app_insights_resource_id = existing_ai_logger.get("app_insights_resource_id")

preflight_rows = []

if existing_ai_logger or app_insights_resource_id or cfg.app_insights_connection_string:
    preflight_rows.append({
        "check": "Application Insights connected",
        "status": "PASS" if existing_ai_logger else "FAIL",
        "detail": (
            f"APIM logger found: {existing_ai_logger['logger_id']}"
            if existing_ai_logger
            else "No APIM logger found yet; configure step can create one from APP_INSIGHTS_* values."
        ),
        "remediation": "Run the Configure section below to ensure the logger and API diagnostic."
    })
else:
    preflight_rows.append({
        "check": "Application Insights connected",
        "status": "FAIL",
        "detail": "No APIM Application Insights logger or APP_INSIGHTS_* values were found.",
        "remediation": "Set APP_INSIGHTS_RESOURCE_ID (or APP_INSIGHTS_NAME) and APP_INSIGHTS_CONNECTION_STRING in .env, then rerun."
    })

try:
    diagnostic = apim.get_api_diagnostic(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_API_ID
    )
except Exception as exc:
    diagnostic = None
    diagnostic_error = str(exc)
else:
    diagnostic_error = ""

llm_settings = ((diagnostic or {}).get("properties", {}) or {}).get("largeLanguageModel")
preflight_rows.append({
    "check": "LLM API logging enabled",
    "status": "PASS" if llm_settings else "FAIL",
    "detail": "API diagnostic has largeLanguageModel settings." if llm_settings else (diagnostic_error or "Demo 2 API diagnostic is not enabled yet."),
    "remediation": "Run the Configure section below; it enables API-scope diagnostics with LLM logging."
})

custom_metric_check = apim.check_custom_metric_dimensions_enabled(app_insights_resource_id)
preflight_rows.append({
    "check": "Custom metrics with dimensions enabled",
    "status": custom_metric_check["status"],
    "detail": custom_metric_check["detail"],
    "remediation": f"{custom_metric_check['remediation']} Portal: {custom_metric_check['portal_url']}"
})

bounded_ok = CLIENT_APP_ALLOW_LIST == {"claims-portal", "analyst-copilot"}
preflight_rows.append({
    "check": "Client sends a bounded x-client-app value",
    "status": "PASS" if bounded_ok else "FAIL",
    "detail": f"Allow-list: {sorted(CLIENT_APP_ALLOW_LIST)}",
    "remediation": "Do not send free-form user, prompt, or tenant strings as metric dimensions."
})

preflight_df = display.show_table(preflight_rows, columns=["check", "status", "detail", "remediation"])

for row in preflight_rows:
    kind = "success" if row["status"] == "PASS" else ("warning" if row["status"] == "MANUAL" else "error")
    display.banner(f"{row['status']}: {row['check']}", kind=kind)


## Configure (policy apply)

This section creates or updates the Demo 2 resources on the existing APIM instance using idempotent ARM `PUT` calls:

- Backend `demo2-openai-backend`
- API `demo2-metering-api`
- `chat/completions` operation
- Product `demo2-metering`
- Subscription `demo2-metering-sub`
- Optional `demo2-aoai-key` named value when `AOAI_KEY` is supplied
- Application Insights logger and API diagnostic when App Insights values are available
- API-scope policy from `policies/demo2-emit-token-metric.xml`


In [ ]:
display.header("Creating backend")
backend = apim.ensure_backend(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    backend_id=DEMO_BACKEND_ID,
    backend_url=cfg.aoai_endpoint,
    description="Demo 2 Azure OpenAI backend",
    protocol="http",
)
display.banner(f"Backend '{DEMO_BACKEND_ID}' ensured.", kind="success")


In [ ]:
display.header("Creating API and operation")
api = apim.ensure_api(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    display_name="Demo 2 - Token Metering (Azure OpenAI)",
    path=DEMO_PATH,
    service_url=cfg.aoai_endpoint.rstrip("/"),
    subscription_required=True,
)

if API_STYLE == "v1":
    OPERATION_URL_TEMPLATE = "/openai/v1/chat/completions"
else:
    OPERATION_URL_TEMPLATE = f"/openai/deployments/{cfg.aoai_deployment}/chat/completions"

operation = apim.ensure_operation(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    operation_id="chat-completions",
    display_name="Chat Completions",
    method="POST",
    url_template=OPERATION_URL_TEMPLATE,
)
display.banner(f"API '{DEMO_API_ID}' and operation '{OPERATION_URL_TEMPLATE}' ensured.", kind="success")


In [ ]:
display.header("Creating product, subscription, and optional key named value")

apim.ensure_product(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    product_id=DEMO_PRODUCT_ID,
    display_name="Demo2-Metering",
    description="Isolated product for the Demo 2 token-metering workshop scenario.",
    subscription_required=True,
    state="published",
)
apim.ensure_product_api_link(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    product_id=DEMO_PRODUCT_ID,
    api_id=DEMO_API_ID,
)
apim.ensure_subscription(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    apim_subscription_id=DEMO_SUBSCRIPTION_ID,
    display_name="Demo2-Metering-Subscription",
    scope=f"/products/{DEMO_PRODUCT_ID}",
)

if cfg.aoai_key:
    apim.ensure_named_value(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        named_value_id=DEMO_NAMED_VALUE_KEY,
        display_name=DEMO_NAMED_VALUE_KEY,
        value=cfg.aoai_key,
        secret=True,
    )
    display.banner("AOAI key named value ensured (key masked).", kind="success")
else:
    display.banner(
        "No AOAI key supplied -- assuming managed identity is configured for this APIM instance to call Azure OpenAI.",
        kind="warning",
    )

display.banner(
    f"Product '{DEMO_PRODUCT_ID}' and subscription '{DEMO_SUBSCRIPTION_ID}' ensured.",
    kind="success",
)


In [ ]:
display.header("Ensuring Application Insights logger and LLM API diagnostic")

if (cfg.app_insights_connection_string or "").strip():
    # APIM requires non-empty credentials on an applicationInsights logger, so
    # always supply the connection string even when resourceId is known. APIM
    # mints a Logger-Credentials--* named value on each PUT; that side effect is
    # expected. Stale entries can be deleted manually, keeping the one referenced
    # by the live logger's credentials.connectionString {{...}} token.
    logger = apim.ensure_logger(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        logger_id=DEMO_LOGGER_ID,
        app_insights_resource_id=app_insights_resource_id,
        app_insights_connection_string=cfg.app_insights_connection_string,
        description="Demo 2 Application Insights logger",
    )
    active_logger_id = DEMO_LOGGER_ID
    display.banner(f"Application Insights logger '{DEMO_LOGGER_ID}' ensured.", kind="success")
elif existing_ai_logger:
    active_logger_id = existing_ai_logger["logger_id"]
    display.banner(f"Using existing APIM Application Insights logger '{active_logger_id}'.", kind="success")
else:
    active_logger_id = None
    display.banner(
        "No Application Insights logger can be created because APP_INSIGHTS_* values are missing. Metrics may still emit, but verification fallback will be limited.",
        kind="warning",
    )

if active_logger_id:
    apim.ensure_api_diagnostic(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        api_id=DEMO_API_ID,
        logger_id=active_logger_id,
    )
    display.banner("API-scope diagnostic with LLM logging ensured.", kind="success")


## Apply the policy at API scope

The policy authenticates to Azure OpenAI with APIM managed identity by default. If `AOAI_KEY` is set, the notebook swaps the managed-identity block for the same API-key fallback pattern used in Demo 1, sourced from the `demo2-aoai-key` named value.


In [ ]:
with open("../policies/demo2-emit-token-metric.xml", encoding="utf-8-sig") as f:
    policy_xml = f.read()

_MI_AUTH_BLOCK = re.compile(
    r"\s*<authentication-managed-identity\b.*?</set-header>", re.DOTALL
)
_API_KEY_BLOCK = (
    '\n    <set-header name="api-key" exists-action="override">'
    "\n      <value>{{demo2-aoai-key}}</value>"
    "\n    </set-header>"
)

if cfg.aoai_key:
    policy_xml, replaced = _MI_AUTH_BLOCK.subn(_API_KEY_BLOCK, policy_xml, count=1)
    if not replaced:
        raise ValueError("Could not find the managed-identity block in the policy XML.")
    display.banner(
        "AOAI key supplied -- the policy will authenticate to the backend with the 'api-key' header (value from demo2-aoai-key).",
        kind="info",
    )
else:
    display.banner(
        "No AOAI key -- the policy authenticates to the backend with the APIM managed identity.",
        kind="info",
    )

print(policy_xml)


In [ ]:
apim.set_api_policy(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
    policy_xml=policy_xml,
)
display.banner("Policy applied at API scope.", kind="success")


## Data-plane call helper

Every call goes through the APIM gateway with the dedicated Demo 2 subscription key. The client helper refuses to send any `x-client-app` value outside the bounded allow-list.


In [ ]:
GATEWAY_URL = apim.get_gateway_url(cfg.subscription_id, cfg.resource_group, cfg.apim_name)
SUBSCRIPTION_KEY = apim.get_subscription_key(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name, DEMO_SUBSCRIPTION_ID
)

print(f"Gateway URL: {GATEWAY_URL}")
print(f"Subscription key: {auth.mask_secret(SUBSCRIPTION_KEY)}")


def _chat_url_and_body(prompt: str, max_tokens: int = 60, stream: bool = False):
    if API_STYLE == "v1":
        url = f"{GATEWAY_URL}/{DEMO_PATH}/openai/v1/chat/completions"
        body = {
            "model": cfg.aoai_deployment,
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        }
    else:
        url = (
            f"{GATEWAY_URL}/{DEMO_PATH}/openai/deployments/"
            f"{cfg.aoai_deployment}/chat/completions"
        )
        body = {
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
        }
    if stream:
        body["stream"] = True
        body["stream_options"] = {"include_usage": True}
    return url, body


def call_chat_completion(prompt: str, client_app: str, max_tokens: int = 60, stream: bool = False):
    """Call Demo 2 and capture token usage returned by the provider."""
    if client_app not in CLIENT_APP_ALLOW_LIST:
        raise ValueError(
            f"x-client-app must be one of {sorted(CLIENT_APP_ALLOW_LIST)}; got {client_app!r}"
        )

    headers = {
        "Ocp-Apim-Subscription-Key": SUBSCRIPTION_KEY,
        "Content-Type": "application/json",
        "x-client-app": client_app,
        "x-demo-run": DEMO_RUN,
    }
    url, body = _chat_url_and_body(prompt, max_tokens=max_tokens, stream=stream)
    params = {} if API_STYLE == "v1" else {"api-version": cfg.aoai_api_version}

    start = time.time()
    response = requests.post(url, headers=headers, params=params, json=body, timeout=90, stream=stream)
    latency_ms = round((time.time() - start) * 1000, 1)

    reply = None
    usage = {}
    if response.status_code == 200 and not stream:
        try:
            payload = response.json()
            reply = payload["choices"][0]["message"]["content"]
            usage = payload.get("usage", {}) or {}
        except Exception:
            reply = None
            usage = {}

    return {
        "status": response.status_code,
        "latency_ms": latency_ms,
        "client_app": client_app,
        "prompt_tokens": usage.get("prompt_tokens"),
        "completion_tokens": usage.get("completion_tokens"),
        "total_tokens": usage.get("total_tokens"),
        "reply": reply,
        "body": response.text[:2000] if not stream else "<streaming response>",
        "url": response.url,
    }


def explain_failure(result):
    status = result["status"]
    display.banner(f"Call did not return 200 (status {status}): {result['body']}", kind="error")
    if status == 404:
        print(f"Requested URL   : {result.get('url')}")
        print(f"AOAI endpoint   : {cfg.aoai_endpoint}")
        print(f"AOAI deployment : {cfg.aoai_deployment}")
        print(f"API style       : {API_STYLE}")
        print("Likely causes: AOAI_ENDPOINT includes a path, AOAI_DEPLOYMENT is wrong, or AOAI_API_STYLE is wrong.")
    elif status in (401, 403):
        print("Likely cause: APIM managed identity lacks 'Cognitive Services OpenAI User', or AOAI_KEY is invalid.")
    elif status == 500:
        print("Likely cause: APIM policy expression/runtime error or managed-identity token acquisition failure.")


## Baseline

Make one call with `x-client-app: claims-portal`, show the provider's token usage, and confirm the policy was in place to emit a metric asynchronously.


In [ ]:
baseline = call_chat_completion("Say hello in one short sentence.", "claims-portal", max_tokens=40)
display.show_table([{
    "status": baseline["status"],
    "latency_ms": baseline["latency_ms"],
    "client_app": baseline["client_app"],
    "prompt_tokens": baseline["prompt_tokens"],
    "completion_tokens": baseline["completion_tokens"],
    "total_tokens": baseline["total_tokens"],
    "reply": baseline["reply"],
}])

if baseline["status"] != 200:
    explain_failure(baseline)
else:
    display.banner("Baseline call succeeded; llm-emit-token-metric will publish token counts asynchronously.", kind="success")


## Demonstrate: 5× `claims-portal` + 3× `analyst-copilot`

Now prove the meter by generating exactly the split from the deck: five calls as `claims-portal` and three calls as `analyst-copilot`. The helper enforces the bounded-header allow-list before each request is sent.


In [ ]:
traffic_plan = ["claims-portal"] * 5 + ["analyst-copilot"] * 3
traffic_log = []

for index, client_app in enumerate(traffic_plan, start=1):
    result = call_chat_completion(
        f"Demo 2 metering call {index}: answer with one concise sentence.",
        client_app,
        max_tokens=50,
    )
    traffic_log.append({
        "request_number": index,
        "client_app": client_app,
        "status": result["status"],
        "latency_ms": result["latency_ms"],
        "prompt_tokens": result["prompt_tokens"],
        "completion_tokens": result["completion_tokens"],
        "total_tokens": result["total_tokens"],
    })
    if result["status"] != 200:
        explain_failure(result)
        break
    time.sleep(1)

traffic_df = display.show_table(traffic_log)
summary_by_client = traffic_df.groupby("client_app")[["prompt_tokens", "completion_tokens", "total_tokens"]].sum().reset_index() if traffic_df is not None else pd.DataFrame()
display.show_table(summary_by_client.to_dict("records"))


## Verify: split total tokens by `ClientApp` in 5-minute bins

Azure Monitor and Application Insights metrics are not synchronous. **Ingestion delay is normal**: if the chart looks empty, wait before debugging. The polling cell below retries with backoff for up to a few minutes.

The preferred path queries Azure Monitor metrics on the APIM resource (`Microsoft.ApiManagement/service`) for namespace `module8`, split by `ClientApp`, filtered to the dedicated Demo 2 APIM subscription. If that namespace is not queryable in your tenant yet, the cell falls back to Application Insights `customMetrics` Kusto query when an App Insights resource id is known.


In [ ]:
APIM_RESOURCE_ID = (
    f"/subscriptions/{cfg.subscription_id}/resourceGroups/{cfg.resource_group}"
    f"/providers/Microsoft.ApiManagement/service/{cfg.apim_name}"
)


def _normalize_metric_name(metric_name: str):
    compact = metric_name.replace(" ", "").replace("_", "").lower()
    if compact in {"prompttokens", "prompttoken"}:
        return "prompt_tokens"
    if compact in {"completiontokens", "completiontoken"}:
        return "completion_tokens"
    if compact in {"totaltokens", "totaltoken"}:
        return "total_tokens"
    return metric_name


def _query_metric_candidates(query_func, *args, **kwargs):
    rows = []
    for canonical_name, candidates in TOKEN_METRIC_CANDIDATES.items():
        for candidate in candidates:
            try:
                candidate_rows = query_func(*args, metric_names=[candidate], **kwargs)
            except ImportError:
                raise
            except Exception as exc:
                print(f"Metric candidate {candidate!r} was not queryable here: {exc}")
                continue
            if candidate_rows:
                for row in candidate_rows:
                    row["metric_name"] = canonical_name
                rows.extend(candidate_rows)
                break
    return rows


# Preflight: llm-emit-token-metric writes dimensioned custom metrics through the
# Application Insights logger. When "Alerting on custom metric dimensions" is off,
# those metrics are dropped at ingestion and the query below can only ever return
# zero rows, even though request/dependency telemetry keeps flowing.
dimension_check = apim.check_custom_metric_dimensions_enabled(app_insights_resource_id)
if dimension_check["status"] == "PASS":
    display.banner(
        f"PASS: custom metric dimensions - {dimension_check['detail']}",
        kind="success",
    )
else:
    display.banner(
        f"{dimension_check['status']}: custom metric dimensions could not be confirmed. "
        f"{dimension_check['detail']} Dimensioned custom metrics (including the token "
        f"metrics below) may be dropped at ingestion until 'Alerting on custom metric "
        f"dimensions' is enabled. {dimension_check['remediation']} "
        f"Usage and estimated costs: {dimension_check['portal_url']}",
        kind="warning",
    )


def fetch_token_metric_rows():
    """Read token metrics from Application Insights customMetrics.

    This is the only correct source: the policy's namespace is a label on an App
    Insights custom metric, not an Azure Monitor metric namespace on the APIM
    resource, so APIM-scoped metric queries cannot return these series.
    """
    query_start = RUN_STARTED_AT - timedelta(minutes=1)
    query_end = datetime.now(timezone.utc) + timedelta(minutes=1)
    rows = _query_metric_candidates(
        apim.query_token_metrics,
        resource_id=APIM_RESOURCE_ID,
        app_insights_resource_id=app_insights_resource_id,
        dimension_name="ClientApp",
        subscription_filter=DEMO_SUBSCRIPTION_ID,
        start_time=query_start,
        end_time=query_end,
    )
    return rows, "Application Insights customMetrics"

# Fail fast: none of the remaining failure modes resolve with time, so verify the
# preconditions before spending ~4 minutes in the retry schedule below.
preflight_failures = []

demo_diagnostic = apim.get_api_diagnostic(
    cfg.subscription_id, cfg.resource_group, cfg.apim_name,
    api_id=DEMO_API_ID,
)
if not demo_diagnostic:
    preflight_failures.append(
        f"The 'applicationinsights' diagnostic does not exist on API '{DEMO_API_ID}', "
        "so llm-emit-token-metric has no logger to emit through. Re-run the Configure "
        "section above."
    )
elif not demo_diagnostic.get("properties", {}).get("metrics"):
    preflight_failures.append(
        "Custom metrics are disabled on the API diagnostic, so metrics emitted by "
        "llm-emit-token-metric are silently discarded. Re-run the Configure section "
        "above, or tick it in the portal under APIs -> the API -> Settings -> "
        "Diagnostics Logs -> Application Insights -> 'Support custom metrics'."
    )

applied_policy_xml = (
    apim.get_api_policy(
        cfg.subscription_id, cfg.resource_group, cfg.apim_name,
        api_id=DEMO_API_ID,
    )
    .get("properties", {})
    .get("value", "")
)
if "llm-emit-token-metric" not in applied_policy_xml:
    preflight_failures.append(
        "The policy applied at API scope does not contain llm-emit-token-metric, so no "
        "token metrics are emitted at all. Re-run the policy cell above."
    )

if preflight_failures:
    for failure in preflight_failures:
        display.banner(failure, kind="error")
    raise RuntimeError(
        "Token metrics cannot be emitted with the current configuration. Fix the issues "
        "above, re-run the call cell, then re-run this cell."
    )

metric_rows = []
metric_source = None
for attempt, sleep_seconds in enumerate([15, 30, 45, 60, 75], start=1):
    metric_rows, metric_source = fetch_token_metric_rows()
    if metric_rows:
        display.banner(f"Token metrics found in {metric_source} on attempt {attempt}.", kind="success")
        break
    display.banner(
        f"No token metrics yet (attempt {attempt}). Waiting {sleep_seconds}s for ingestion delay before retrying.",
        kind="warning",
    )
    time.sleep(sleep_seconds)

if not metric_rows:
    display.banner(
        "No token metrics were returned. The API diagnostic, its custom metrics setting, "
        "and the applied policy were all verified above, so the remaining cause is that "
        "custom metric dimensions are disabled on the Application Insights resource and "
        "the dimensioned metrics emitted by llm-emit-token-metric are discarded at "
        "ingestion - enable 'Alerting on custom metric dimensions' at "
        f"{dimension_check['portal_url']} and re-run the call cell. Note that "
        "service-level request and dependency telemetry keeps flowing regardless, so "
        "seeing telemetry in Application Insights does not prove token metrics arrive.",
        kind="error",
    )

metric_df = display.show_table(metric_rows, columns=["timestamp", "metric_name", "dimension_value", "total"])
display.plot_token_series_by_dimension(metric_rows, dimension_name="ClientApp", metric_name="total_tokens")


## Acceptance: PASS 01 / PASS 02 / PASS 03

We assert and display the three acceptance criteria from the deck:

- **PASS 01:** Two `ClientApp` series appear.
- **PASS 02:** Prompt + completion reconcile to total.
- **PASS 03:** Subscription filters cleanly isolate chargeback.


In [ ]:
def _metric_sum(rows, metric_name, client_app=None):
    total = 0
    for row in rows:
        if row.get("metric_name") != metric_name:
            continue
        if client_app and row.get("dimension_value") != client_app:
            continue
        total += float(row.get("total") or 0)
    return total

seen_client_apps = {
    row.get("dimension_value")
    for row in metric_rows
    if row.get("metric_name") == "total_tokens" and float(row.get("total") or 0) > 0
}

prompt_total = _metric_sum(metric_rows, "prompt_tokens")
completion_total = _metric_sum(metric_rows, "completion_tokens")
total_tokens = _metric_sum(metric_rows, "total_tokens")
reconciles = total_tokens > 0 and abs((prompt_total + completion_total) - total_tokens) <= 0.01
subscription_isolated = total_tokens > 0 and bool(metric_rows)

acceptance_rows = [
    {
        "criterion": "PASS 01: Two ClientApp series appear",
        "status": "PASS" if {"claims-portal", "analyst-copilot"}.issubset(seen_client_apps) else "FAIL",
        "evidence": f"Seen ClientApp series: {sorted(seen_client_apps)}",
    },
    {
        "criterion": "PASS 02: Prompt + completion reconcile to total",
        "status": "PASS" if reconciles else "FAIL",
        "evidence": f"prompt={prompt_total}, completion={completion_total}, total={total_tokens}",
    },
    {
        "criterion": "PASS 03: Subscription filters cleanly isolate chargeback",
        "status": "PASS" if subscription_isolated else "FAIL",
        "evidence": f"Query filtered on Subscription ID == {DEMO_SUBSCRIPTION_ID}; total_tokens={total_tokens}",
    },
]

display.show_table(acceptance_rows, columns=["criterion", "status", "evidence"])
for row in acceptance_rows:
    display.banner(f"{row['status']}: {row['criterion']}", kind="success" if row["status"] == "PASS" else "error")


> **Streaming caveat**
>
> For streaming responses, request token usage from the provider when supported: `stream_options: {"include_usage": true}`. Interrupted streams can produce incomplete counts because the final usage event may never arrive at the gateway or client.
>
> The main Demo 2 flow uses non-streaming calls so the prompt, completion, and total token counts reconcile deterministically during a live workshop.


In [ ]:
# Optional mini-check: build the streaming request body without sending it.
# Uncomment the final line if you want to demonstrate a streaming call live.
stream_url, stream_body = _chat_url_and_body(
    "Stream one short sentence and include usage if supported.",
    max_tokens=40,
    stream=True,
)
display.show_table([{
    "stream": stream_body.get("stream"),
    "stream_options": stream_body.get("stream_options"),
    "caveat": "Interrupted streams can produce incomplete counts.",
}])
# streaming_result = call_chat_completion("Stream one short sentence.", "claims-portal", max_tokens=40, stream=True)


## Summary: what you saw → which policy/knob made it happen

| What you saw | Which policy/knob made it happen |
| --- | --- |
| Token metrics emitted into `module8` | `llm-emit-token-metric namespace="module8"` |
| Default identity grain | `<dimension name="API ID" />` and `<dimension name="Subscription ID" />` |
| Business chargeback grain | `<dimension name="ClientApp" value="@(context.Request.Headers.GetValueOrDefault(&quot;x-client-app&quot;,&quot;unknown&quot;))" />` |
| Only two business series | Client-side allow-list: `claims-portal`, `analyst-copilot` |
| Subscription filters isolate chargeback | Dedicated APIM subscription `demo2-metering-sub` plus metric filter on `Subscription ID` |
| Prompt + completion reconcile to total | Provider token usage surfaced as prompt, completion, and total token metric grains |

> **Cardinality / cost callout:** Keep custom dimensions bounded. APIM supports a limited cardinality budget (≤5 custom dimensions, ~100 values each). Do not use user IDs, prompt text, session IDs, request IDs, or other unbounded values as metric dimensions.


## Reset / Cleanup guidance

Demo 2 leaves its APIM resources in place because later demos reuse the same APIM instance and because metric ingestion is useful for follow-up exploration.

Safe reset options:

- Re-run the notebook: all resource creation and policy application cells are idempotent.
- Regenerate `DEMO_RUN` below for request traceability in logs. This does **not** change the metric dimensions, by design, because `ClientApp` must remain bounded.
- For final workshop cleanup, remove the demo APIs/products/subscriptions together after Demo 4.


In [ ]:
DEMO_RUN = uuid.uuid4().hex[:8]
config.persist_demo_run(DEMO_RUN)
print(f"New DEMO_RUN suffix for request traceability: {DEMO_RUN}")
display.banner("Demo 2 reset value persisted. Metric dimensions remain bounded and unchanged.", kind="success")
